In [ ]:
import random
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree, distance
from sklearn.cluster import KMeans
from operator import itemgetter

from cargar_archivos import transformar_kml, guardar_kmz, exportar_rutas_kml_por_pareja
from utils import ordenar_ruta
from generar_planillas import generar_todas_las_planillas



## Preparación:

In [ ]:
path_mapa = "Mapa_Independencia.kml"
path_marco = "20260506 Marco Muestral (actualizado).xlsx"
path_curso = "20260430 Lista Curso Sol208.xlsx"

df = transformar_kml(path_mapa)
print(df)

df_marco = pd.read_excel(path_marco, sheet_name=1)
df_curso = pd.read_excel(path_curso, sheet_name=1) #Cargamos ambas en la segunda página

#separamos en dos distintos df

df_A = df[df["Estrato"] == "Estrato A"].copy()
df_B = df[df["Estrato"] == "Estrato B"].copy()


    Folio    Estrato    Latitud   Longitud  \
0       1  Estrato A -33.419934 -70.670726   
1       2  Estrato A -33.415846 -70.673154   
2       3  Estrato A -33.422955 -70.659353   
3       4  Estrato A -33.420495 -70.670362   
4       5  Estrato A -33.422559 -70.660571   
..    ...        ...        ...        ...   
600   607  Estrato B -33.422705 -70.671264   
601   608  Estrato B -33.420468 -70.671561   
602   609  Estrato B -33.418983 -70.674432   
603   610  Estrato B -33.415534 -70.672025   
604   326  Estrato B -33.420981 -70.675121   

                    Descripcion_Original  
0             Salomon Sack 380 Depto. 11  
1            16 Norte 1138 Independencia  
2                Colon 1450 Depto. A 104  
3                       Sara Gajardo 960  
4             Pasaje San Rafael Casa 19B  
..                                   ...  
600                 Gamero 2710 Depto 13  
601    Salomon Sack 410 Blok D Depto. 32  
602              Longitudinal 3 Casa 571  
603              

## Etapa I:

Tal como se describio en el informe del trabajo, la primera etapa del diseño muestral es buscar la K-Medias del Estrato A. Para hacerlo utilizaremos el objeto KMeans del paquete sklearn:

In [ ]:
#Creamos un objeto para obtener las K-Medias
kmedias = KMeans(n_clusters=33, random_state=42, n_init= "auto") #random_state = seed n_init = iteraciones

#Creamos las predicciones en base las coordenadas de los puntos del Estrato A
df_A["Cluster"] = kmedias.fit_predict(df_A[["Latitud", "Longitud"]]) 

centroides = kmedias.cluster_centers_ # retorna la lista de los 33 centroides



## Etapa 2:

Ya teniendo nuestros K-Centroides, creamos el KD-Tree para mappear nuestras direcciones del Estrato B

In [ ]:
#Creamos un arbol KD para mapear cada uno de los puntos del estrato B por su ubicación geográfica
arbol_B = cKDTree(df_B[["Latitud", "Longitud"]]) 

"""
Creamos una tabla de hash que tenga todos los indices de las casas disponibles.
El motivo por el cual se habla de indices y no de folios, es porque nuestro arbol
no maneja datos de los folios, solo de sus indices. Entonces, para saber si es que
están disponibles o no, debemos revisar por el número otorgado por el arbol
"""
folios_disponibles = set(range(len(df_B))) 


muestra_final = [df_A]

for cluster_id in range(33): #Iteramos del 0 al 32 (Indices de los clusters)
    if 10 - len(df_A[df_A["Cluster"] == cluster_id]) > 0: #10 - (total de nodos en el clusters a revisar)
        dist, indices = arbol_B.query(centroides[cluster_id], k=70) #Sacamos las 90 direcciones más cercanas

        nodos_guardados = []
        for idx in indices:
            if idx in folios_disponibles: #Busqueda O(1)
                nodos_guardados.append(idx)
                folios_disponibles.remove(idx)

                if len(nodos_guardados) + len(df_A[df_A["Cluster"] == cluster_id]) == 10:
                    break
        df_B_nuevo = df_B.iloc[nodos_guardados].copy() #copiamos todos los nodos que hayamos guardados
        df_B_nuevo["Cluster"] = cluster_id #Le asignamos el número de su cluster

        muestra_final.append(df_B_nuevo)

*muestra_final* es una lista de DataFrames que contienen todas las direcciones agregadas por cluster. Utilizando pd.concat(), podemos concatenar todas estas listas en un único DataFrame:

In [ ]:
df_final = pd.concat(muestra_final, ignore_index=True)

print(f"\nTamaño de la muestra final: {len(df_final)} direcciones")

print("\nDistribución por Cluster:")
print(df_final["Cluster"].value_counts().sort_index())


Tamaño de la muestra final: 330 direcciones

Distribución por Cluster:
Cluster
0     10
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
12    10
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21    10
22    10
23    10
24    10
25    10
26    10
27    10
28    10
29    10
30    10
31    10
32    10
Name: count, dtype: int64


## Etapa 3:

Ya teniendo la clusterización completa, iniciamos a procesar los datos del curso. Iniciamos normalizando los números de grupos, y buscando la cantidad de casos bordes a analizar:

In [ ]:
#Reducimos todos los valores por 10 porque por algún motivo los grupos parten del 10 (?)
df_curso["Grupo"] = df_curso["Grupo"] - 10

df_curso["Grupo"] = df_curso["Grupo"].rank(method="dense").astype(int) #También reordenamos los valores que partan del 1 y lleguen al 17
df_curso = df_curso.sort_values(by="Grupo")



total_por_grupo = df_curso["Grupo"].value_counts()

de_2 = total_por_grupo[total_por_grupo < 3]
de_3 = total_por_grupo[total_por_grupo == 3]
de_4 = total_por_grupo[total_por_grupo == 4]
de_5plus = total_por_grupo[total_por_grupo > 4]


#Para implementar correctamente nuestro algoritmo, debemos revisar el total de grupos que no son de
#4 personas ya que eso se representaria como un caso borde.

print(f"---- CASOS BORDE ----")
print(f"Cantidad de grupos de 2 o menos personas: {len(de_2)}\n")
print(f"Cantidad de grupos de 3 personas: {len(de_3)}\n") 
print(f"Cantidad de grupos de 5 o más personas: {len(de_5plus)}\n")
print(df_curso)

---- CASOS BORDE ----
Cantidad de grupos de 2 o menos personas: 0

Cantidad de grupos de 3 personas: 2

Cantidad de grupos de 5 o más personas: 0

    Sección  N° Apellido Paterno Apellido Materno                    Nombres  \
22        1  23            SILVA           GARCÍA  AMANDA MERCEDES DEL PILAR   
23        1  24            SOLÍS           GORMAZ            CATALINA ANDREA   
5         1   6        CONTRERAS         HONORATO                KAREN SOFÍA   
26        1  27         TRUJILLO         CALDERÓN             VICENTE ANDRÉS   
0         1   1           AGUAYO          JIMÉNEZ           LUCIANO SALVADOR   
..      ...  ..              ...              ...                        ...   
32        2   3          ÁLVAREZ          ÁLVAREZ         CATALINA MARGARITA   
55        2  26           O'RYAN            PÉREZ                    BEATRIZ   
54        2  25            NUÑEZ         MARTÍNEZ            ANAHI VALENTINA   
52        2  23         MELÉNDEZ            ROJAS    

Entonces, sabemos que tenemos 17 grupos en total, donde 2 de ellos son de 3 personas y el resto (15) son de 4 personas. Siguiendo esta lógica, dividiremos cada grupo por la mitad aleatoriamente y serán asignados un nuevo número de pareja, y para ambos grupos impares, dos de ellos de cada uno serán seleccionados al azar, y cada uno de los restantes formarán una pareja entre ellos

In [ ]:
lista_total_grupos = df_curso.groupby("Grupo").size()
grupos_de_4 = lista_total_grupos[lista_total_grupos == 4].index.tolist()
grupos_de_3 = lista_total_grupos[lista_total_grupos == 3].index.tolist()

df_curso["ID_Pareja"] = 0 #Inicializamos la columna
contador = 1 #Nos va a ser útil para asignar el ID de las parejas
pareja_faltante = []

np.random.seed(124)

for id_grupo in grupos_de_4:
    idxs = df_curso[df_curso["Grupo"] == id_grupo].index.tolist()
    np.random.shuffle(idxs) #Randomizamos los indices del grupo en una lista

    #Los primeros dos del nuestra lista randomizada irán serán la segunda pareja
    df_curso.loc[idxs[:2], "ID_Pareja"] = contador 
    contador += 1

    #Los últimos dos la segunda
    df_curso.loc[idxs[2:], "ID_Pareja"] = contador
    contador += 1

for id_grupo in grupos_de_3:
    idxs = df_curso[df_curso["Grupo"] == id_grupo].index.tolist()
    np.random.shuffle(idxs)

    #Solo calculamos la primera pareja en los grupos de a 3
    df_curso.loc[idxs[:2], "ID_Pareja"] = contador
    contador += 1

    #Agregamos la persona restante a la lista de rezagados
    pareja_faltante.append(idxs[2])


df_curso.loc[pareja_faltante, "ID_Pareja"] = contador #se agregan al df

print(df_curso)

    Sección  N° Apellido Paterno Apellido Materno                    Nombres  \
22        1  23            SILVA           GARCÍA  AMANDA MERCEDES DEL PILAR   
23        1  24            SOLÍS           GORMAZ            CATALINA ANDREA   
5         1   6        CONTRERAS         HONORATO                KAREN SOFÍA   
26        1  27         TRUJILLO         CALDERÓN             VICENTE ANDRÉS   
0         1   1           AGUAYO          JIMÉNEZ           LUCIANO SALVADOR   
..      ...  ..              ...              ...                        ...   
32        2   3          ÁLVAREZ          ÁLVAREZ         CATALINA MARGARITA   
55        2  26           O'RYAN            PÉREZ                    BEATRIZ   
54        2  25            NUÑEZ         MARTÍNEZ            ANAHI VALENTINA   
52        2  23         MELÉNDEZ            ROJAS              GONZALO AMARO   
63        2  34           TORRES       EYZAGUIRRE          VICENTE FRANCISCO   

    G  Grupo  ID_Pareja  
22  1      1 

Finalmente, ahora que tenemos nuestras 33 parejas, debemos asignarles sus clusters correspondiente. Para asegurar que cada grupo tenga una muestra con la mayor hetereogenidad posible, usaremos un algoritmo en el cual por cada pareja le selecciona algún centroide aleatorio, y luego buscará el centroide con la mayor distancia euclidiana del seleccionado disponible para asignarle a la otra pareja del grupo. Como bien sabemos sin embargo, existen dos casos borde, y para ellos buscaremos cual de los 3 centroides restantes tiene una mayor distancia eucladiana a los otros dos.

In [ ]:
centroides_disponibles = list(range(33)) #Enumeramos todos los centroides en un lista
asignacion_parejas = {}
grupos_de_4 = de_4.tolist() 

"""
Por como funciona nuestra DF con parejas, los grupos de 4 personas tienen asignados
Los ID de pareja del 1 al 30, entonces, para que funcione con solo saber el número del
grupo usamos enumerate(), el cual ordena los valores en tuplas de menor a mayor, por lo
cual se veria asi (indice, grupo_id). Entonces, al obtener el valor de su indice,
al usar la lógica de (indice * 2) + 1 podemos acceder a la primera pareja del grupo 1.

¿Por qué no lo hacemos simplemente con el número de grupo? Debido a que el grupo 6 y el
grupo 4 son los de 3 personas (y sus ID de parejas son 31, 32 y 33, ya que fueron añadidos
al final), esa lógica deja de aplicar después del grupo 4, por ejemplo, los id de pareja del
grupo 5 son 6 y 7.
"""

lista_grupos = enumerate(grupos_de_4)

random.seed(123)

for i, grupo_id in lista_grupos:
    pareja_A = (i * 2) + 1  #lógica explicada previamente
    pareja_B = (i * 2) + 2  
    
    cluster_A = random.choice(centroides_disponibles)
    asignacion_parejas[pareja_A] = cluster_A
    centroides_disponibles.remove(cluster_A)
    
    coord_A = centroides[cluster_A]
    cluster_B = None
    max_distancia = -1
    
    for c in centroides_disponibles:
        """
        Buscamos dentro de todos los centroides_disponibles, cual es que tiene mayor
        distancia al escogido. (Este algoritmo es similar a SelectionSort)
        """
        dist = distance.euclidean(coord_A, centroides[c])
        if dist > max_distancia: 
            max_distancia = dist
            cluster_B = c
            
    asignacion_parejas[pareja_B] = cluster_B
    centroides_disponibles.remove(cluster_B)

"""
Ahora nos queda encontrar los clusters para nuestros últimos 3 grupos.
"""
c_1, c_2, c_3 = centroides_disponibles

dist_c1 = (distance.euclidean(centroides[c_1], centroides[c_2]) + distance.euclidean(centroides[c_1], centroides[c_3]), c_1)
dist_c2 = (distance.euclidean(centroides[c_2], centroides[c_1]) + distance.euclidean(centroides[c_2], centroides[c_3]), c_2)
dist_c3 = (distance.euclidean(centroides[c_3], centroides[c_1]) + distance.euclidean(centroides[c_3], centroides[c_2]), c_3)

cluster_mixto = max(dist_c1, dist_c2, dist_c3, key=itemgetter(0))

asignacion_parejas[33] = cluster_mixto[1]
centroides_disponibles.remove(cluster_mixto[1])

asignacion_parejas[31] = centroides_disponibles[0]
asignacion_parejas[32] = centroides_disponibles[1]

In [ ]:
#Creamos una nueva columna que sea el nombre completo (Nombre + apellido)
df_curso["Nombre_Completo"] = df_curso["Nombres"].str.split().str[0] + " " + df_curso["Apellido Paterno"] 

#Transformamos a que el nombre de la pareja sea la composición de ambos nombres completos juntos.
df_parejas = df_curso.groupby("ID_Pareja").agg({
    "Nombre_Completo": lambda x: " y ".join(x),  
    "Grupo": lambda x: " / ".join(x.astype(str).unique()) 
}).reset_index()

df_parejas.rename(columns={"Nombre": "Nombres_Estudiantes"}, inplace=True)
cluster_a_pareja = {cluster_id: pareja for pareja, cluster_id in asignacion_parejas.items()}
df_final["ID_Pareja"] = df_final["Cluster"].map(cluster_a_pareja)

df_final = pd.merge(df_final, df_parejas, on="ID_Pareja", how="left")

print(df_final[["Folio", "Cluster", "ID_Pareja", "Grupo", "Nombre_Completo"]])

    Folio  Cluster  ID_Pareja Grupo                     Nombre_Completo
0       1       15         11     8     COLOMBA MESINA y LEÓN FERNÁNDEZ
1       2        8         31     4       TOMÁS LEVA y CRISTÓBAL MORENO
2       3        2          5     3  ESPERANZA INFANTE y MARÍA VALDIVIA
3       4       15         11     8     COLOMBA MESINA y LEÓN FERNÁNDEZ
4       5        2          5     3  ESPERANZA INFANTE y MARÍA VALDIVIA
..    ...      ...        ...   ...                                 ...
325   341       32         27    16    FLORENCIA ALFARO y BELÉN POLITIS
326   228       32         27    16    FLORENCIA ALFARO y BELÉN POLITIS
327   198       32         27    16    FLORENCIA ALFARO y BELÉN POLITIS
328   411       32         27    16    FLORENCIA ALFARO y BELÉN POLITIS
329   222       32         27    16    FLORENCIA ALFARO y BELÉN POLITIS

[330 rows x 5 columns]


Ahora, debemos guardar cada uno de las 5 mejores viviendas de reserva para cada pareja. Ahora, se hará en order inverso al sorteo original, para asegurar que la última pareja no salga igual de desaventajada.

In [ ]:
reservas = []

for cluster_id in reversed(range(33)): #Iteramos del 32 al 0 (Orden inverso)
    dist, indices = arbol_B.query(centroides[cluster_id], k=len(folios_disponibles)) #Sacamos las 130 direcciones más cercanas

    nodos_guardados = []
    for idx in indices:
        if idx in folios_disponibles: #Busqueda O(1)
            nodos_guardados.append(idx)
            folios_disponibles.remove(idx)

            if len(nodos_guardados) == 3: #si ya encontró 3 puntos a guardar, termina su iteración
                break
    df_B_nuevo = df_B.iloc[nodos_guardados].copy() #copiamos todos los nodos que hayamos guardados
    df_B_nuevo["Cluster"] = cluster_id #Le asignamos el número de su cluster

    reservas.append(df_B_nuevo)

df_reservas = pd.concat(reservas)

Por último, debemos calcular una ruta óptima entre las direcciones del clúster. El algoritmo que la calculará, siempre partirá del punto más "sureño" del clúster, e irá buscando iterativamente el punto más cercano de él.

In [ ]:
rutas = []

for cluster_id in sorted(df_final["Cluster"].unique()): #Iteramos sobre cada ID de clúster
    df_cluster = df_final[df_final["Cluster"] == cluster_id] #Sacamos todos sus filas
    
    #El algoritmo de ordenar_ruta puede ser encontrado en utils.py
    df_cluster_ordenado = ordenar_ruta(df_cluster) #Las ordenamos
    rutas.append(df_cluster_ordenado)

df_enrutado = pd.concat(rutas, ignore_index=True)



## Guardado:

Ahora que tenemos todos los datos necesarios, los guardamos en sus archivos correspondientes. Primero partimos generado todas las planillas, luego guardamos todo en un .xlsx, y finalmente guardamos el mapa final y los mapas por pareja

In [ ]:
generar_todas_las_planillas(df_enrutado, df_reservas) #Creamos todas las planillas

#Creamos un excel con todos los dfs
with pd.ExcelWriter("Trabajo 2-Grupo 14.xlsx") as writer:
    df_curso.to_excel(writer, sheet_name="Datos Curso")
    df_final.to_excel(writer, sheet_name="Datos Clústers")
    df_reservas.to_excel(writer, sheet_name="Folios de Reserva")

guardar_kmz(df_final) #Y guardamos el KMZ del mapa final
exportar_rutas_kml_por_pareja(df_enrutado) #Guardamos también un mapa donde se enseñe la ruta óptima


Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/18/planilla_pareja_18.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/15/planilla_pareja_15.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/5/planilla_pareja_5.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/1/planilla_pareja_1.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/6/planilla_pareja_6.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/29/planilla_pareja_29.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/13/planilla_pareja_13.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/24/planilla_pareja_24.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/31/planilla_pareja_31.pdf
Generando PDF usando WeasyPrint...
✅ PDF generado con éxito en: Parejas/3/planilla_pareja_3.pdf
Generando PDF usando WeasyPr